# Design and Implementation of a Text Preprocessing Pipeline

### K. J. Somaiya School of Engineering, Mumbai-77
### (Somaiya Vidyavihar University)
### Department of Computer Engineering

| | |
|---|---|
| **Class** | MTech -- Semester 1 |
| **Course** | NLP (Natural Language Processing) |
| **Term** | July -- August -- November 2026 |
| **Assignment** | Design and Implementation of a Text Preprocessing Pipeline |
| **Student Name** | *Aryan Sanjay Gole* |
| **Roll No. / PRN** | *16030726006* |
| **Batch** | ** |
| **Date of Submission** | *24/08/2026* |

---

**Dataset used:** IMDB Movie Reviews Dataset (50,000 reviews)

**Notebook type:** Google Colab -- runs top to bottom without manual editing

>

## Introduction

### Aim

To design and implement a complete NLP text preprocessing pipeline in Python and analyze the effect of different preprocessing techniques on real-world textual data.

### Problem Statement

Raw, real-world text is noisy: it contains HTML markup, inconsistent capitalization, punctuation, numbers, URLs, email addresses, contractions, and stopwords that carry little independent meaning on their own. Before such text can be used for downstream NLP tasks (sentiment analysis, classification, information retrieval, etc.), it must be cleaned and normalized into a consistent representation -- without discarding information that changes meaning (for example, negation words such as *not* and *never*, which matter a great deal for a sentiment dataset).

This assignment requires selecting a real-world text dataset with sufficient textual variation and building a configurable preprocessing pipeline that performs:

- Dataset exploration
- Sentence segmentation
- Tokenization (compared across multiple approaches)
- Cleaning and normalization
- Stopword handling
- Stemming and lemmatization
- A single reusable, configurable `preprocess_text()` function
- Before/after analysis of the effect of preprocessing
- A critical comparison of aggressive vs. meaning-preserving preprocessing on domain-sensitive (clinical) text

### Dataset Description

**Domain selected:** Product / Movie Reviews -- the **IMDB Dataset of 50,000 Movie Reviews**, a widely used, publicly available sentiment-analysis dataset.

| Property | Value |
|---|---|
| Total documents | 50,000 |
| Columns | `review` (text), `sentiment` (`positive` / `negative`) |
| Class balance | 25,000 positive / 25,000 negative (perfectly balanced) |
| Missing values | None |
| Average document length | ~ 235 words per review (word-boundary tokens) |
| Vocabulary size before preprocessing | ~ 112,000 unique word-like tokens |
| Reviews containing HTML `<br />` tags | ~ 58% |

This dataset comfortably exceeds the minimum 50-sample requirement and contains exactly the kind of textual variation the assignment asks for: HTML tags, punctuation, numbers/ratings (e.g. "10/10"), informal contractions ("didn't", "it's"), inconsistent capitalization, and occasional special characters -- all of which are demonstrated and handled in the tasks below.

## Google Colab Setup

This section prepares the Colab runtime:

1. Import all required libraries .
2. Define the configuration constants used throughout the notebook -- , e.g. to change `START_SAMPLE` / `END_SAMPLE` **as per our own choice we can choice a start and end sample value**
3. Download the NLTK resources the pipeline depends on (`punkt`, `stopwords`, `wordnet`, the POS tagger, etc.) -- this runs automatically and only downloads what is missing.

In [40]:

import os
import re
import sys
import html
import string
from collections import Counter

import pandas as pd
from IPython.display import display, Markdown

try:
    import nltk
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nltk"], check=True)
    import nltk

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag

pd.set_option("display.max_colwidth", 150)

print("Libraries imported successfully.")

Libraries imported successfully.


In [41]:

# --------------------------------------------------------------
# Samples are 1-indexed and inclusive of both ends, matching the
# (e.g. START_SAMPLE=101, END_SAMPLE=200
# selects the 100 documents from row 101 to row 200).
# --------------------------------------------------------------
START_SAMPLE = 101
END_SAMPLE = 200

RECOMMENDED_MIN_SAMPLES = 50

DISPLAY_DOCUMENTS = 3

TRANSFORMATION_DOCUMENTS = 5

MAX_DISPLAY_TOKENS = 80

MOUNT_GOOGLE_DRIVE = False

print("Configuration loaded.")
print(f"START_SAMPLE = {START_SAMPLE}, END_SAMPLE = {END_SAMPLE}")

Configuration loaded.
START_SAMPLE = 101, END_SAMPLE = 200


In [42]:


stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

In [43]:

def print_header(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)


def print_subheader(title):
    print("\n" + "-" * 70)
    print(title)
    print("-" * 70)


def print_success(message):
    print(f"\n[SUCCESS] {message}")


def print_warning(message):
    print(f"\n[WARNING] {message}")


def print_error(message):
    print(f"\n[ERROR] {message}")

In [44]:

def setup_nltk_resources():

    print_header("NLP RESOURCE CHECK")

    resources = [
        ("tokenizers/punkt", "punkt"),
        ("tokenizers/punkt_tab", "punkt_tab"),
        ("corpora/stopwords", "stopwords"),
        ("corpora/wordnet", "wordnet"),
        ("corpora/omw-1.4", "omw-1.4"),
        ("taggers/averaged_perceptron_tagger_eng",
         "averaged_perceptron_tagger_eng")
    ]

    all_available = True

    for resource_path, download_name in resources:

        try:
            nltk.data.find(resource_path)

            print(
                f"[SUCCESS] NLTK resource available: "
                f"{download_name}"
            )

        except LookupError:

            print(
                f"[WARNING] NLTK resource missing: "
                f"{download_name}"
            )

            print(
                f"Downloading {download_name}..."
            )

            try:
                success = nltk.download(
                    download_name,
                    quiet=False
                )

                if success:
                    print(
                        f"[SUCCESS] Downloaded: "
                        f"{download_name}"
                    )
                else:
                    print_error(
                        f"Unable to download: "
                        f"{download_name}"
                    )
                    all_available = False

            except Exception as error:

                print_error(
                    f"Error downloading {download_name}"
                )

                print(f"Details: {error}")

                all_available = False

    return all_available


nltk_ready = setup_nltk_resources()

if not nltk_ready:
    print_error(
        "One or more required NLTK resources could not be downloaded. "
        "Check your Colab runtime's internet access and re-run this cell "
        "before continuing."
    )


NLP RESOURCE CHECK
[SUCCESS] NLTK resource available: punkt
[SUCCESS] NLTK resource available: punkt_tab
[SUCCESS] NLTK resource available: stopwords
[WARNING] NLTK resource missing: wordnet
[SUCCESS] Downloaded: wordnet
[WARNING] NLTK resource missing: omw-1.4
[SUCCESS] Downloaded: omw-1.4
[SUCCESS] NLTK resource available: averaged_perceptron_tagger_eng


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [45]:
if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    print_success("Google Drive mounted at /content/drive.")
else:
    print("Skipping Google Drive mount (MOUNT_GOOGLE_DRIVE = False).")
    print("The dataset will be loaded directly from the Colab session instead.")

Skipping Google Drive mount (MOUNT_GOOGLE_DRIVE = False).
The dataset will be loaded directly from the Colab session instead.


In [46]:

CANDIDATE_FILENAMES = [
    "IMDB Dataset.csv",
    "IMDB_Dataset.csv",
    "imdb_dataset.csv",
    "imdb dataset.csv",
]

CANDIDATE_DIRECTORIES = [
    os.getcwd(),
    os.path.join(os.getcwd(), "dataset"),
    "/content",
    "/content/dataset",
    "/content/drive/MyDrive",
    "/content/drive/MyDrive/dataset",
]


def find_dataset_file():
    """Search common Colab locations/name variants for the IMDB dataset."""


    for directory in CANDIDATE_DIRECTORIES:
        if not os.path.isdir(directory):
            continue
        for name in CANDIDATE_FILENAMES:
            candidate = os.path.join(directory, name)
            if os.path.exists(candidate):
                return candidate

    for directory in [os.getcwd(), "/content"]:
        if not os.path.isdir(directory):
            continue
        for fname in os.listdir(directory):
            normalized = fname.lower().replace("_", " ").strip()
            if normalized == "imdb dataset.csv":
                return os.path.join(directory, fname)

    return None


def validate_dataset():

    print_header(
        "STEP 1 - DATASET SELECTION AND VALIDATION"
    )

    print("Selected dataset:")
    print("IMDB Dataset.csv")

    print(f"\nSearching for the dataset in: {', '.join(CANDIDATE_DIRECTORIES)}")

    path = find_dataset_file()

    if path is None:

        print_error(
            "IMDB Dataset.csv was not found automatically in any of the "
            "usual Colab locations."
        )


        try:
            from google.colab import files
            print(
                "\nRunning inside Google Colab: please upload "
                "'IMDB Dataset.csv' using the widget below."
            )
            uploaded = files.upload()
            for fname in uploaded.keys():
                if fname.lower().endswith(".csv"):
                    path = os.path.join(os.getcwd(), fname)
                    break
        except ImportError:
            pass

    if path is None or not os.path.exists(path):

        print_error(
            "IMDB Dataset.csv could not be located or uploaded."
        )

        return None

    print("\nDataset path found:")
    print(path)

    print_success(
        "STEP 1 completed successfully."
    )

    return path


IMDB_DATASET_PATH = validate_dataset()

if IMDB_DATASET_PATH is None:
    raise FileNotFoundError(
        "IMDB Dataset.csv could not be found. Upload it to the Colab "
        "session (Files panel) or place it in the working directory, "
        "then re-run this cell."
    )


STEP 1 - DATASET SELECTION AND VALIDATION
Selected dataset:
IMDB Dataset.csv

Searching for the dataset in: /content, /content/dataset, /content, /content/dataset, /content/drive/MyDrive, /content/drive/MyDrive/dataset

Dataset path found:
/content/IMDB Dataset.csv

[SUCCESS] STEP 1 completed successfully.


In [47]:

def load_dataset():

    print_header(
        "STEP 2 - DATASET LOADING"
    )

    try:

        print("Loading IMDB dataset...")

        df = pd.read_csv(
            IMDB_DATASET_PATH
        )

        required_columns = [
            "review",
            "sentiment"
        ]

        missing_columns = [
            column
            for column in required_columns
            if column not in df.columns
        ]

        if missing_columns:

            print_error(
                f"Required columns are missing: "
                f"{missing_columns}"
            )

            return None

        df["review"] = (
            df["review"]
            .fillna("")
            .astype(str)
        )

        df["sentiment"] = (
            df["sentiment"]
            .fillna("")
            .astype(str)
        )

        print_success(
            "Dataset loaded successfully."
        )

        print(
            f"\nTotal rows loaded: {len(df)}"
        )

        print(
            f"Total columns: {len(df.columns)}"
        )

        print("\nAvailable columns:")

        for column in df.columns:

            print(f" - {column}")

        return df

    except Exception as error:

        print_error(
            "Unable to load the dataset."
        )

        print(
            f"\nDetails: {error}"
        )

        return None


df = load_dataset()

if df is None:
    raise RuntimeError(
        "Program stopped because dataset loading failed. See the error "
        "details printed above."
    )


STEP 2 - DATASET LOADING
Loading IMDB dataset...

[SUCCESS] Dataset loaded successfully.

Total rows loaded: 50000
Total columns: 2

Available columns:
 - review
 - sentiment


In [48]:


def basic_tokens(text):

    return re.findall(
        r"\b[\w']+\b",
        text.lower()
    )


print(f"Number of documents: {len(df)}")

print("\nSample documents:")
display(df.head(3))

_lengths = [len(basic_tokens(text)) for text in df["review"]]
_average_length = sum(_lengths) / len(_lengths) if _lengths else 0
print(f"\nAverage document length: {_average_length:.2f} words")

_vocabulary = set()
for text in df["review"]:
    _vocabulary.update(basic_tokens(text))
print(f"Vocabulary size before preprocessing: {len(_vocabulary)} unique tokens")

Number of documents: 50000

Sample documents:


,review,sentiment
0,"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened ...",positive
1,"A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometim...",positive
2,"I thought this was a wonderful way to spend time on a too hot summer weekend, sitting in the air conditioned theater and watching a light-hearted ...",positive



Average document length: 234.97 words
Vocabulary size before preprocessing: 112358 unique tokens


## Sample Range Selection

The original script collected `START` and `END` sample numbers interactively via `input()`. Interactive prompts cannot run inside a notebook that must execute automatically from top to bottom, so this notebook uses the **configurable `START_SAMPLE` / `END_SAMPLE` variables** defined in the Configuration cell near the top instead -- the same validation rules (range bounds, minimum recommended sample count) are still applied, just without blocking on manual entry.

The selected range (documents `START_SAMPLE`-`END_SAMPLE`) is used throughout the notebook for the detailed, per-document demonstrations (Tasks 1, 2, 3, 4 and 6). The full 50,000-document dataset is still used wherever the assignment asks for whole-dataset statistics (Tasks 8 and 9).

In [49]:

def select_sample_range(df, start, end):

    print_header(
        "SAMPLE RANGE SELECTION"
    )

    total_samples = len(df)

    print(
        f"Total available samples: "
        f"{total_samples}"
    )

    print(
        "\nThe selected range will be used "
        "for detailed demonstrations."
    )

    print(
        "\nRecommended:"
    )

    print(
        "Select approximately 80-100 samples."
    )

    print(
        "Example: START = 101, END = 200"
    )

    if start < 1:
        print_warning(f"START ({start}) must be at least 1. Adjusted to 1.")
        start = 1

    if end > total_samples:
        print_warning(
            f"END ({end}) cannot exceed {total_samples}. "
            f"Adjusted to {total_samples}."
        )
        end = total_samples

    if start > end:
        print_error(
            f"START ({start}) cannot be greater than END ({end}). "
            f"Swapping the two values."
        )
        start, end = end, start

    selected_count = (
        end - start + 1
    )

    if selected_count < RECOMMENDED_MIN_SAMPLES:

        print_warning(
            f"You selected only "
            f"{selected_count} samples."
        )

        print(
            "At least 50 samples are "
            "recommended for this practical."
        )

        print(
            "Continuing automatically with this range since the "
            "notebook must run without manual input."
        )

    selected_df = df.iloc[
        start - 1:end
    ].copy()

    print(
        "\nSelected sample range:"
    )

    print(f"START: {start}")
    print(f"END:   {end}")

    print(
        f"\nNumber of selected samples: "
        f"{len(selected_df)}"
    )

    print_success(
        "Sample range selected successfully."
    )

    return selected_df, start, end


selected_df, start, end = select_sample_range(df, START_SAMPLE, END_SAMPLE)


SAMPLE RANGE SELECTION
Total available samples: 50000

The selected range will be used for detailed demonstrations.

Recommended:
Select approximately 80-100 samples.
Example: START = 101, END = 200

Selected sample range:
START: 101
END:   200

Number of selected samples: 100

[SUCCESS] Sample range selected successfully.


# Task 1 -- Dataset Exploration

**Assignment requirement:** load the dataset and display the number of documents, sample raw documents, average document length, and the number of unique words before preprocessing; briefly describe the major types of noise observed.

This calls the original `explore_dataset()` function (STEP 3 in the source script) unchanged. It reports these statistics twice: once for the **complete dataset** (all 50,000 reviews) and once for the **selected sample range** (`START_SAMPLE`-`END_SAMPLE`) that the rest of the notebook works with in detail.

In [50]:

def explore_dataset(
    df,
    selected_df,
    start,
    end
):

    print_header(
        "STEP 3 - DATASET EXPLORATION"
    )


    print("COMPLETE DATASET")

    print("-" * 70)

    print(
        f"Number of documents: {len(df)}"
    )

    sentiment_counts = (
        df["sentiment"]
        .value_counts()
    )

    print(
        "\nSentiment distribution:"
    )

    for sentiment, count in sentiment_counts.items():

        percentage = (
            count / len(df)
        ) * 100

        print(
            f" - {sentiment}: "
            f"{count} "
            f"({percentage:.2f}%)"
        )

    lengths = [
        len(basic_tokens(text))
        for text in df["review"]
    ]

    average_length = (
        sum(lengths) / len(lengths)
        if lengths else 0
    )

    print(
        f"\nAverage document length: "
        f"{average_length:.2f} words"
    )

    print(
        "\nCalculating vocabulary..."
    )

    vocabulary = set()

    for text in df["review"]:

        vocabulary.update(
            basic_tokens(text)
        )

    print(
        "Number of unique words before "
        f"preprocessing: {len(vocabulary)}"
    )

    # --------------------------------------------------------
    # SELECTED DATA
    # --------------------------------------------------------

    print_subheader(
        "SELECTED SAMPLE RANGE"
    )

    print(
        f"Selected range: "
        f"{start} - {end}"
    )

    print(
        f"Selected documents: "
        f"{len(selected_df)}"
    )

    selected_lengths = [
        len(basic_tokens(text))
        for text in selected_df["review"]
    ]

    selected_average = (
        sum(selected_lengths)
        / len(selected_lengths)
        if selected_lengths else 0
    )

    print(
        f"Average selected document length: "
        f"{selected_average:.2f} words"
    )

    selected_vocabulary = set()

    for text in selected_df["review"]:

        selected_vocabulary.update(
            basic_tokens(text)
        )

    print(
        f"Unique words in selected samples: "
        f"{len(selected_vocabulary)}"
    )

    # --------------------------------------------------------
    # RAW DOCUMENTS
    # --------------------------------------------------------

    print_subheader(
        "SAMPLE RAW DOCUMENTS"
    )

    for position in range(
        min(DISPLAY_DOCUMENTS, len(selected_df))
    ):

        row = selected_df.iloc[position]

        sample_number = (
            start + position
        )

        print(
            f"\n--- Sample {sample_number} ---"
        )

        print(
            f"Sentiment: "
            f"{row['sentiment']}"
        )

        print(
            "Raw review:"
        )

        preview = row["review"][:1000]

        if len(row["review"]) > 1000:
            preview += "..."

        print(preview)

    # --------------------------------------------------------
    # NOISE OBSERVATION
    # --------------------------------------------------------

    print_subheader(
        "MAJOR TYPES OF TEXTUAL NOISE OBSERVED"
    )

    print(
        "1. HTML tags such as <br />"
    )

    print(
        "2. Punctuation and repeated punctuation"
    )

    print(
        "3. Extra whitespace"
    )

    print(
        "4. Numbers and rating expressions"
    )

    print(
        "5. Special characters"
    )

    print(
        "6. Contractions such as don't, it's, "
        "and didn't"
    )

    print(
        "7. Capitalization differences"
    )

    print(
        "8. Informal movie-review expressions"
    )

    print_success(
        "STEP 3 completed successfully."
    )

    return {
        "full_vocabulary": vocabulary,
        "selected_vocabulary": selected_vocabulary,
        "average_length": average_length,
        "selected_average_length": selected_average
    }


exploration_stats = explore_dataset(df, selected_df, start, end)


STEP 3 - DATASET EXPLORATION
COMPLETE DATASET
----------------------------------------------------------------------
Number of documents: 50000

Sentiment distribution:
 - positive: 25000 (50.00%)
 - negative: 25000 (50.00%)

Average document length: 234.97 words

Calculating vocabulary...
Number of unique words before preprocessing: 112358

----------------------------------------------------------------------
SELECTED SAMPLE RANGE
----------------------------------------------------------------------
Selected range: 101 - 200
Selected documents: 100
Average selected document length: 246.64 words
Unique words in selected samples: 5060

----------------------------------------------------------------------
SAMPLE RAW DOCUMENTS
----------------------------------------------------------------------

--- Sample 101 ---
Sentiment: positive
Raw review:
This short film that inspired the soon-to-be full length feature - Spatula Madness - is a hilarious piece that contends against similar car

**Observations**

- The dataset is a perfectly balanced binary sentiment classification set (25,000 / 25,000), so no class-imbalance correction is needed before modeling.
- The 8 noise types listed above are exactly the textual variation the assignment's Task 1 asks students to identify -- HTML markup, punctuation, whitespace, numbers/ratings, special characters, contractions, capitalization, and informal review language all appear directly in the raw text.
- Compare the "complete dataset" average document length/vocabulary to the "selected sample range" figures printed above: the selected range is a small slice of the corpus, so its vocabulary is naturally a small fraction of the full 50,000-document vocabulary -- this is expected sampling behavior, not an error.

# Task 2 -- Sentence Segmentation

**Assignment requirement:** implement sentence segmentation, display the result for at least 3 sample documents, and discuss how abbreviations (e.g. *Dr., Mr., U.S.*) can make sentence-boundary detection difficult.

`segment_sentences()` wraps NLTK's `sent_tokenize()` with a simple regex fallback, then `demonstrate_sentence_segmentation()` (STEP 4 in the source script) runs it on the first 3 documents of the selected range and on the exact "Dr. Sharma" abbreviation example the assignment gives.

In [51]:

def segment_sentences(text):

    try:

        return sent_tokenize(text)

    except Exception:


        return re.split(
            r"(?<=[.!?])\s+",
            text
        )


def demonstrate_sentence_segmentation(
    selected_df,
    start
):

    print_header(
        "STEP 4 - SENTENCE SEGMENTATION"
    )

    print(
        "Sentence segmentation divides each "
        "document into individual sentences."
    )

    for position in range(
        min(3, len(selected_df))
    ):

        row = selected_df.iloc[position]

        sample_number = (
            start + position
        )

        sentences = segment_sentences(
            row["review"]
        )

        print(
            f"\n--- Sample {sample_number} ---"
        )

        print(
            f"Total sentences detected: "
            f"{len(sentences)}"
        )

        for number, sentence in enumerate(
            sentences[:10],
            start=1
        ):

            print(
                f"{number}. {sentence}"
            )

        if len(sentences) > 10:

            print(
                "... additional sentences omitted"
            )


    print_subheader(
        "DIFFICULT CASE: ABBREVIATION"
    )

    abbreviation_example = (
        "Dr. Sharma discussed NLP techniques. "
        "Students implemented them in Python."
    )

    print(
        "\nOriginal text:"
    )

    print(abbreviation_example)

    abbreviation_sentences = (
        segment_sentences(
            abbreviation_example
        )
    )

    print(
        "\nDetected sentences:"
    )

    for number, sentence in enumerate(
        abbreviation_sentences,
        start=1
    ):

        print(
            f"{number}. {sentence}"
        )

    print(
        "\nObservation:"
    )

    print(
        "Abbreviations such as Dr., Mr., "
        "U.S., etc. contain periods that "
        "do not necessarily indicate sentence "
        "boundaries."
    )

    print(
        "A sentence tokenizer must distinguish "
        "these abbreviation periods from actual "
        "sentence-ending punctuation."
    )

    print_success(
        "STEP 4 completed successfully."
    )

    return True


demonstrate_sentence_segmentation(selected_df, start)


STEP 4 - SENTENCE SEGMENTATION
Sentence segmentation divides each document into individual sentences.

--- Sample 101 ---
Total sentences detected: 9
1. This short film that inspired the soon-to-be full length feature - Spatula Madness - is a hilarious piece that contends against similar cartoons yielding multiple writers.
2. The short film stars Edward the Spatula who after being fired from his job, joins in the fight against the evil spoons.
3. This premise allows for some funny content near the beginning, but is barely present for the remainder of the feature.
4. This film's 15-minute running time is absorbed by some odd-ball comedy and a small musical number.
5. Unfortunately not much else lies below it.
6. The plot that is set up doesn't really have time to show.
7. But it's surely follows it plot better than many high-budget Hollywood films.
8. This film is worth watching at least a few times.
9. Take it for what it is, and don't expect a deep story.

--- Sample 102 ---
Total se

True

**Observations**

- NLTK's `sent_tokenize()` (Punkt) uses a statistical model of common abbreviations, so it correctly keeps **"Dr. Sharma discussed NLP techniques."** as a single sentence instead of splitting right after "Dr." -- exactly the case the assignment asks students to test.
- Movie reviews also contain informal punctuation patterns (e.g. "!!!", "...", ALL-CAPS exclamations) that can produce more sentence boundaries than a strict grammatical reading would suggest; this is normal for user-generated review text, not a segmentation failure.
- The regex fallback (`re.split(r"(?<=[.!?])\s+", text)`) only exists so the pipeline degrades gracefully if an NLTK resource is temporarily unavailable -- it has **no** abbreviation awareness, which is precisely why it is a fallback rather than the primary method.

# Task 3 -- Tokenization Comparison

**Assignment requirement:** perform word tokenization, compare at least two tokenization approaches, and identify at least 3 cases where they produce different results. Example given in the assignment: `"Natural Language Processing is interesting!"` -> `['Natural', 'Language', 'Processing', 'is', 'interesting', '!']`.

`compare_tokenizers()` (STEP 5) compares **three** approaches -- Python's `str.split()`, NLTK's `word_tokenize()`, and a custom regular-expression tokenizer -- which exceeds the assignment's minimum of two, on a sentence built to contain punctuation, a dollar amount, a ratio (`10/10`), and an email address (the same kind of case NLTK handles the way the assignment's own example expects: punctuation becomes its own token).

In [52]:
def simple_split_tokenizer(text):

    return text.split()


def nltk_tokenizer(text):

    try:

        return word_tokenize(text)

    except Exception:

        return re.findall(
            r"\w+|[^\w\s]",
            text
        )


def regex_tokenizer(text):

    return re.findall(
        r"\w+(?:'\w+)?|[^\w\s]",
        text
    )


def display_tokens(tokens):

    if len(tokens) > MAX_DISPLAY_TOKENS:

        return (
            str(tokens[:MAX_DISPLAY_TOKENS])
            + " ..."
        )

    return str(tokens)


def compare_tokenizers(
    selected_df,
    start
):

    print_header(
        "STEP 5 - TOKENIZATION COMPARISON"
    )

    comparison_text = (
        "I really loved this movie! "
        "It cost $10, and I gave it 10/10. "
        "Email test@example.com."
    )

    print(
        "Demonstration text:"
    )

    print(comparison_text)

    split_tokens = (
        simple_split_tokenizer(
            comparison_text
        )
    )

    nltk_tokens = (
        nltk_tokenizer(
            comparison_text
        )
    )

    regex_tokens = (
        regex_tokenizer(
            comparison_text
        )
    )

    print_subheader(
        "TOKENIZATION RESULTS"
    )

    print(
        "\n1. Simple split()"
    )

    print(
        display_tokens(
            split_tokens
        )
    )

    print(
        "\n2. NLTK word_tokenize()"
    )

    print(
        display_tokens(
            nltk_tokens
        )
    )

    print(
        "\n3. Regular-expression tokenizer"
    )

    print(
        display_tokens(
            regex_tokens
        )
    )

    print_subheader(
        "THREE IMPORTANT DIFFERENCES"
    )

    print(
        "1. Punctuation:"
    )

    print(
        "split() usually keeps punctuation "
        "attached to words, while NLTK can "
        "produce punctuation as separate tokens."
    )

    print(
        "\n2. Contractions:"
    )

    print(
        "NLTK can separate apostrophe-based "
        "contractions differently from split()."
    )

    print(
        "\n3. Numbers and special expressions:"
    )

    print(
        "Expressions such as 10/10, $10 and "
        "email@example.com can be handled "
        "differently depending on the tokenizer."
    )



    print_subheader(
        "DATASET TOKENIZATION EXAMPLE"
    )

    if len(selected_df) > 0:

        sample_text = (
            selected_df.iloc[0]["review"]
        )

        print(
            f"\nSample {start}:"
        )

        print(
            "\nSimple split():"
        )

        print(
            display_tokens(
                simple_split_tokenizer(
                    sample_text
                )
            )
        )

        print(
            "\nNLTK tokenizer:"
        )

        print(
            display_tokens(
                nltk_tokenizer(
                    sample_text
                )
            )
        )

    print_success(
        "STEP 5 completed successfully."
    )

    return True


compare_tokenizers(selected_df, start)


STEP 5 - TOKENIZATION COMPARISON
Demonstration text:
I really loved this movie! It cost $10, and I gave it 10/10. Email test@example.com.

----------------------------------------------------------------------
TOKENIZATION RESULTS
----------------------------------------------------------------------

1. Simple split()
['I', 'really', 'loved', 'this', 'movie!', 'It', 'cost', '$10,', 'and', 'I', 'gave', 'it', '10/10.', 'Email', 'test@example.com.']

2. NLTK word_tokenize()
['I', 'really', 'loved', 'this', 'movie', '!', 'It', 'cost', '$', '10', ',', 'and', 'I', 'gave', 'it', '10/10', '.', 'Email', 'test', '@', 'example.com', '.']

3. Regular-expression tokenizer
['I', 'really', 'loved', 'this', 'movie', '!', 'It', 'cost', '$', '10', ',', 'and', 'I', 'gave', 'it', '10', '/', '10', '.', 'Email', 'test', '@', 'example', '.', 'com', '.']

----------------------------------------------------------------------
THREE IMPORTANT DIFFERENCES
-------------------------------------------------------

True

In [53]:

_demo_text = (
    "I really loved this movie! "
    "It cost $10, and I gave it 10/10. "
    "Email test@example.com."
)

tokenizer_comparison_df = pd.DataFrame([
    {
        "Tokenizer": "Simple split()",
        "Token Count": len(simple_split_tokenizer(_demo_text)),
        "Tokens": display_tokens(simple_split_tokenizer(_demo_text))
    },
    {
        "Tokenizer": "NLTK word_tokenize()",
        "Token Count": len(nltk_tokenizer(_demo_text)),
        "Tokens": display_tokens(nltk_tokenizer(_demo_text))
    },
    {
        "Tokenizer": "Regex tokenizer",
        "Token Count": len(regex_tokenizer(_demo_text)),
        "Tokens": display_tokens(regex_tokenizer(_demo_text))
    },
])

display(tokenizer_comparison_df)

,Tokenizer,Token Count,Tokens
0,Simple split(),15,"['I', 'really', 'loved', 'this', 'movie!', 'It', 'cost', '$10,', 'and', 'I', 'gave', 'it', '10/10.', 'Email', 'test@example.com.']"
1,NLTK word_tokenize(),22,"['I', 'really', 'loved', 'this', 'movie', '!', 'It', 'cost', '$', '10', ',', 'and', 'I', 'gave', 'it', '10/10', '.', 'Email', 'test', '@', 'exampl..."
2,Regex tokenizer,26,"['I', 'really', 'loved', 'this', 'movie', '!', 'It', 'cost', '$', '10', ',', 'and', 'I', 'gave', 'it', '10', '/', '10', '.', 'Email', 'test', '@',..."


**Observations**

- `$10` and `10/10` are handled differently across tokenizers: `split()` keeps them as single whitespace-delimited chunks, while NLTK and the regex tokenizer are more likely to separate the numeric part from surrounding punctuation.
- The email address `test@example.com` and the sentence-final period after it are another good example: `split()` keeps `test@example.com.` glued together as one token, while NLTK typically separates the trailing sentence period from it.
- Exclamation marks and commas stay attached to the previous word with `split()`, but appear as their own tokens with NLTK's `word_tokenize()` -- on its own this is one of the "at least 3 cases" the assignment asks for, and the dollar-amount, ratio, and email cases above provide two more.

# Task 4 -- Cleaning and Normalization

**Assignment requirement:** implement text normalization/cleaning (lowercasing, whitespace, punctuation, URLs, emails, special characters, numbers, stopwords) *without blindly removing everything* -- preprocessing choices should be justified for the selected dataset.

This is `clean_text()` and its component functions (STEP 6), followed by stopword handling with negation preservation (STEP 7). Every cleaning operation is independently toggleable (`lowercase`, `remove_html_tags`, `remove_url`, `remove_email`, `remove_punct`, `remove_num`, `normalize_space`), and stopword removal defaults to **keeping** `not`, `no`, `nor`, and `never` -- a deliberate, dataset-specific decision, since this is a sentiment dataset where negation flips meaning.

In [54]:

def lowercase_text(text):

    return text.lower()


def remove_html(text):

    text = html.unescape(text)

    return re.sub(
        r"<[^>]+>",
        " ",
        text
    )


def remove_urls(text):

    return re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text
    )


def remove_emails(text):

    return re.sub(
        r"\b[A-Za-z0-9._%+-]+@"
        r"[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
        " ",
        text
    )


def remove_numbers(text):

    return re.sub(
        r"\b\d+(?:\.\d+)?\b",
        " ",
        text
    )


def remove_punctuation(text):

    return text.translate(
        str.maketrans(
            "",
            "",
            string.punctuation
        )
    )


def normalize_whitespace(text):

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


def clean_text(
    text,
    lowercase=True,
    remove_html_tags=True,
    remove_url=True,
    remove_email=True,
    remove_punct=True,
    remove_num=True,
    normalize_space=True
):

    if lowercase:

        text = lowercase_text(text)

    if remove_html_tags:

        text = remove_html(text)

    if remove_url:

        text = remove_urls(text)

    if remove_email:

        text = remove_emails(text)

    if remove_punct:

        text = remove_punctuation(text)

    if remove_num:

        text = remove_numbers(text)

    if normalize_space:

        text = normalize_whitespace(text)

    return text


def demonstrate_cleaning(
    selected_df,
    start
):

    print_header(
        "STEP 6 - TEXT NORMALIZATION AND CLEANING"
    )

    print(
        "The standard cleaning pipeline performs:"
    )

    print(
        "1. Lowercasing"
    )

    print(
        "2. HTML tag removal"
    )

    print(
        "3. URL removal"
    )

    print(
        "4. Email removal"
    )

    print(
        "5. Punctuation removal"
    )

    print(
        "6. Number removal"
    )

    print(
        "7. Whitespace normalization"
    )

    demonstration_text = (
        "<br /> I REALLY loved this movie! "
        "Visit https://example.com for more "
        "information. Email test@example.com. "
        "It received 10/10 ratings."
    )

    print_subheader(
        "CONTROLLED CLEANING DEMONSTRATION"
    )

    print(
        "\nOriginal:"
    )

    print(demonstration_text)

    text = lowercase_text(
        demonstration_text
    )

    print(
        "\nAfter lowercase:"
    )

    print(text)

    text = remove_html(text)

    print(
        "\nAfter HTML removal:"
    )

    print(text)

    text = remove_urls(text)

    print(
        "\nAfter URL removal:"
    )

    print(text)

    text = remove_emails(text)

    print(
        "\nAfter email removal:"
    )

    print(text)

    text = remove_punctuation(text)

    print(
        "\nAfter punctuation removal:"
    )

    print(text)

    text = remove_numbers(text)

    print(
        "\nAfter number removal:"
    )

    print(text)

    text = normalize_whitespace(text)

    print(
        "\nAfter whitespace normalization:"
    )

    print(text)

    print(
        "\nFinal cleaned text:"
    )

    print(
        clean_text(
            demonstration_text
        )
    )


    print_subheader(
        "DATASET CLEANING DEMONSTRATION"
    )

    for position in range(
        min(3, len(selected_df))
    ):

        row = selected_df.iloc[position]

        sample_number = (
            start + position
        )

        original = row["review"]

        cleaned = clean_text(
            original
        )

        print(
            f"\nSAMPLE {sample_number}"
        )

        print(
            "\nBEFORE:"
        )

        print(
            original[:700]
        )

        if len(original) > 700:
            print("...")

        print(
            "\nAFTER:"
        )

        print(
            cleaned[:700]
        )

        if len(cleaned) > 700:
            print("...")

        print(
            f"\nCharacters before: "
            f"{len(original)}"
        )

        print(
            f"Characters after: "
            f"{len(cleaned)}"
        )

    print_success(
        "STEP 6 completed successfully."
    )

    return True


demonstrate_cleaning(selected_df, start)


STEP 6 - TEXT NORMALIZATION AND CLEANING
The standard cleaning pipeline performs:
1. Lowercasing
2. HTML tag removal
3. URL removal
4. Email removal
5. Punctuation removal
6. Number removal
7. Whitespace normalization

----------------------------------------------------------------------
CONTROLLED CLEANING DEMONSTRATION
----------------------------------------------------------------------

Original:
<br /> I REALLY loved this movie! Visit https://example.com for more information. Email test@example.com. It received 10/10 ratings.

After lowercase:
<br /> i really loved this movie! visit https://example.com for more information. email test@example.com. it received 10/10 ratings.

After HTML removal:
  i really loved this movie! visit https://example.com for more information. email test@example.com. it received 10/10 ratings.

After URL removal:
  i really loved this movie! visit   for more information. email test@example.com. it received 10/10 ratings.

After email removal:
  i real

True

In [55]:


def get_stopwords():

    words = set(
        stopwords.words("english")
    )

    return words


def remove_stopwords(
    tokens,
    preserve_negation=True
):

    stop_words = get_stopwords()

    if preserve_negation:

        stop_words -= {
            "not",
            "no",
            "nor",
            "never"
        }

    return [
        token
        for token in tokens
        if token.lower() not in stop_words
    ]


def demonstrate_stopword_handling():

    print_header(
        "STEP 7 - STOPWORD HANDLING"
    )

    example = (
        "This movie is not very good "
        "and I never recommend it."
    )

    tokens = nltk_tokenizer(
        example
    )

    print(
        "\nOriginal tokens:"
    )

    print(tokens)

    normal_result = remove_stopwords(
        tokens,
        preserve_negation=True
    )

    print(
        "\nAfter stopword removal "
        "with negation preservation:"
    )

    print(normal_result)

    aggressive_result = remove_stopwords(
        tokens,
        preserve_negation=False
    )

    print(
        "\nAfter aggressive stopword removal:"
    )

    print(aggressive_result)

    print(
        "\nObservation:"
    )

    print(
        "For movie sentiment analysis, words "
        "such as 'not', 'no' and 'never' can "
        "change the meaning of a sentence."
    )

    print(
        "Therefore the main IMDB pipeline "
        "preserves these negation words."
    )

    print_success(
        "STEP 7 completed successfully."
    )


demonstrate_stopword_handling()


STEP 7 - STOPWORD HANDLING

Original tokens:
['This', 'movie', 'is', 'not', 'very', 'good', 'and', 'I', 'never', 'recommend', 'it', '.']

After stopword removal with negation preservation:
['movie', 'not', 'good', 'never', 'recommend', '.']

After aggressive stopword removal:
['movie', 'good', 'never', 'recommend', '.']

Observation:
For movie sentiment analysis, words such as 'not', 'no' and 'never' can change the meaning of a sentence.
Therefore the main IMDB pipeline preserves these negation words.

[SUCCESS] STEP 7 completed successfully.


**Observations**

- The controlled demonstration text deliberately contains an HTML tag, a URL, an email address, mixed capitalization, and a `10/10` rating -- exactly the noise categories identified in Task 1 -- so each cleaning stage's effect is visible step by step.
- For the assignment's own example (`Contact NLP-Team@gmail.com for NLP Course 2026!!!`), this pipeline's ordering (lowercase -> HTML -> URL -> email -> punctuation -> numbers -> whitespace) strips the email address and the year number, collapses the repeated `!!!`, and lowercases the remaining words -- producing a clean, normalized phrase in the same spirit as the assignment's worked example.
- **Stopword decision for this dataset:** `not`, `no`, `nor`, and `never` are deliberately *kept* (`preserve_negation=True`) because IMDB reviews rely on negation to flip sentiment (*"not good"* vs. *"good"*). The aggressive variant (`preserve_negation=False`) is shown above only for comparison and is intentionally **not** used as this pipeline's default.

# Task 5 -- Stemming

**Assignment requirement:** apply a stemming algorithm (Porter or Snowball), test it on `connect/connected/connecting/connection`, `study/studies/studying/studied`, and `compute/computer/computing/computation`, present a before/after table, and identify dataset examples where stemming produces unnatural words.

This uses the **Porter Stemmer** (`stem_tokens()`, STEP 8) on exactly the word list the assignment itself provides.

In [56]:


def stem_tokens(tokens):

    return [
        stemmer.stem(token)
        for token in tokens
    ]


def demonstrate_stemming():

    print_header(
        "STEP 8 - STEMMING"
    )

    examples = [
        "connect",
        "connected",
        "connecting",
        "connection",
        "study",
        "studies",
        "studying",
        "studied",
        "compute",
        "computer",
        "computing",
        "computation"
    ]

    print(
        "\nPorter Stemmer results:"
    )

    print(
        f"{'Original Word':<20}"
        f"{'Stemmed Word':<20}"
    )

    print(
        "-" * 40
    )

    for word in examples:

        stem = stemmer.stem(
            word
        )

        print(
            f"{word:<20}"
            f"{stem:<20}"
        )

    print(
        "\nObservation:"
    )

    print(
        "Stemming is rule-based and may "
        "produce truncated or linguistically "
        "unnatural forms."
    )

    print_success(
        "STEP 8 completed successfully."
    )


demonstrate_stemming()


STEP 8 - STEMMING

Porter Stemmer results:
Original Word       Stemmed Word        
----------------------------------------
connect             connect             
connected           connect             
connecting          connect             
connection          connect             
study               studi               
studies             studi               
studying            studi               
studied             studi               
compute             comput              
computer            comput              
computing           comput              
computation         comput              

Observation:
Stemming is rule-based and may produce truncated or linguistically unnatural forms.

[SUCCESS] STEP 8 completed successfully.


In [57]:


_stemming_examples = [
    "connect", "connected", "connecting", "connection",
    "study", "studies", "studying", "studied",
    "compute", "computer", "computing", "computation"
]

stemming_df = pd.DataFrame({
    "Original Word": _stemming_examples,
    "Stemmed Word": stem_tokens(_stemming_examples)
})

display(stemming_df)

,Original Word,Stemmed Word
0,connect,connect
1,connected,connect
2,connecting,connect
3,connection,connect
4,study,studi
5,studies,studi
6,studying,studi
7,studied,studi
8,compute,comput
9,computer,comput


**Observations**

- All four members of the *connect* family reduce to the same stem, and all four members of the *compute* family reduce toward `comput` -- useful for information retrieval, where searches for "computer" and "computing" should be able to match the same documents.
- The *study* family is the assignment's own textbook example of over-stemming: `studies` -> `studi`, which is not a real English word. Porter's suffix-stripping rules are purely mechanical and do not check a dictionary, so this kind of truncation is expected, not a bug.
- Within the selected dataset sample (see Task 1's raw documents and Task 6's dataset examples below), look for `-ing`, `-tion`, and `-ness` words that get shortened past a recognizable root in the same way -- that is the general pattern to look for when identifying "which dataset words does stemming turn into unnatural words".

# Task 6 -- Lemmatization

**Assignment requirement:** apply lemmatization, compare original/stemmed/lemmatized forms, and find at least 5 examples from the dataset where stemming and lemmatization behave differently.

`lemmatize_tokens()` is POS-aware: it runs NLTK's `pos_tag()` over the tokens first, converts each Penn Treebank tag to a WordNet part-of-speech via `get_wordnet_pos()`, and only then calls `WordNetLemmatizer.lemmatize()` -- this is what lets a word like "better" lemmatize to "good" (an adjective-specific WordNet exception) instead of being left unchanged. `demonstrate_lemmatization()` (STEP 9) compares this against Porter stemming on the assignment's own example words, and `find_dataset_stem_lemma_examples()` then searches the selected sample range for real cases where the two techniques disagree -- grouped here since it is this task's dataset deliverable.

Two small utility functions used by the full pipeline (`filter_word_tokens`, `tokenize_clean_text`) are introduced here too, since the dataset search below is the first place they are needed; Task 7 reuses them as part of the complete `preprocess_text()` pipeline.

In [58]:

def get_wordnet_pos(tag):

    if tag.startswith("J"):
        return "a"

    if tag.startswith("V"):
        return "v"

    if tag.startswith("N"):
        return "n"

    if tag.startswith("R"):
        return "r"

    return "n"


def lemmatize_tokens(tokens):

    if not tokens:
        return []

    try:

        tagged_tokens = pos_tag(
            tokens
        )

    except Exception:

        tagged_tokens = [
            (token, "NN")
            for token in tokens
        ]

    result = []

    for token, tag in tagged_tokens:

        wordnet_pos = get_wordnet_pos(
            tag
        )

        lemma = lemmatizer.lemmatize(
            token,
            pos=wordnet_pos
        )

        result.append(
            lemma
        )

    return result




def demonstrate_lemmatization():

    print_header(
        "STEP 9 - LEMMATIZATION"
    )

    examples = [
        "studies",
        "running",
        "better",
        "computers",
        "connected",
        "children",
        "mice",
        "cars"
    ]

    tokens = examples

    stems = stem_tokens(
        tokens
    )

    lemmas = lemmatize_tokens(
        tokens
    )

    print(
        f"\n{'Original':<20}"
        f"{'Stemmed':<20}"
        f"{'Lemmatized':<20}"
    )

    print(
        "-" * 60
    )

    for original, stem, lemma in zip(
        tokens,
        stems,
        lemmas
    ):

        print(
            f"{original:<20}"
            f"{stem:<20}"
            f"{lemma:<20}"
        )

    print_subheader(
        "STEMMING VS LEMMATIZATION"
    )

    print(
        "Stemming removes or modifies word "
        "endings using heuristic rules."
    )

    print(
        "Lemmatization attempts to return the "
        "dictionary/base form of the word."
    )

    print(
        "POS information is used so that "
        "words such as running can be reduced "
        "toward their correct base form."
    )

    print_success(
        "STEP 9 completed successfully."
    )


demonstrate_lemmatization()


STEP 9 - LEMMATIZATION

Original            Stemmed             Lemmatized          
------------------------------------------------------------
studies             studi               study               
running             run                 run                 
better              better              good                
computers           comput              computer            
connected           connect             connect             
children            children            child               
mice                mice                mouse               
cars                car                 car                 

----------------------------------------------------------------------
STEMMING VS LEMMATIZATION
----------------------------------------------------------------------
Stemming removes or modifies word endings using heuristic rules.
Lemmatization attempts to return the dictionary/base form of the word.
POS information is used so that words such as running can be

In [59]:


_lemma_examples = [
    "studies", "running", "better", "computers",
    "connected", "children", "mice", "cars"
]

lemmatization_df = pd.DataFrame({
    "Original": _lemma_examples,
    "Stemmed": stem_tokens(_lemma_examples),
    "Lemmatized": lemmatize_tokens(_lemma_examples)
})

display(lemmatization_df)

,Original,Stemmed,Lemmatized
0,studies,studi,study
1,running,run,run
2,better,better,good
3,computers,comput,computer
4,connected,connect,connect
5,children,children,child
6,mice,mice,mouse
7,cars,car,car


In [60]:


def tokenize_clean_text(text):

    return nltk_tokenizer(
        text
    )


def filter_word_tokens(tokens):

    return [
        token
        for token in tokens
        if re.search(
            r"[A-Za-z]",
            token
        )
    ]


def find_dataset_stem_lemma_examples(
    selected_df
):

    print_header(
        "STEMMING VS LEMMATIZATION - DATASET EXAMPLES"
    )

    found = []

    for text in selected_df["review"]:

        cleaned = clean_text(
            text
        )

        tokens = filter_word_tokens(
            nltk_tokenizer(
                cleaned
            )
        )

        tokens = remove_stopwords(
            tokens,
            preserve_negation=True
        )

        stems = stem_tokens(
            tokens
        )

        lemmas = lemmatize_tokens(
            tokens
        )

        for original, stem, lemma in zip(
            tokens,
            stems,
            lemmas
        ):

            if (
                stem != lemma
                and len(original) > 3
                and len(stem) > 2
            ):

                example = (
                    original,
                    stem,
                    lemma
                )

                if example not in found:

                    found.append(
                        example
                    )

            if len(found) >= 10:
                break

        if len(found) >= 10:
            break

    print(
        f"{'Original':<20}"
        f"{'Stemmed':<20}"
        f"{'Lemmatized':<20}"
    )

    print(
        "-" * 60
    )

    for original, stem, lemma in found:

        print(
            f"{original:<20}"
            f"{stem:<20}"
            f"{lemma:<20}"
        )

    print(
        "\nAt least five examples are displayed "
        "where stemming and lemmatization behave "
        "differently."
    )

    return found


dataset_stem_lemma_examples = find_dataset_stem_lemma_examples(selected_df)

if dataset_stem_lemma_examples:
    display(pd.DataFrame(
        dataset_stem_lemma_examples,
        columns=["Original", "Stemmed", "Lemmatized"]
    ))


STEMMING VS LEMMATIZATION - DATASET EXAMPLES
Original            Stemmed             Lemmatized          
------------------------------------------------------------
inspired            inspir              inspire             
soontobe            soontob             soontobe            
feature             featur              feature             
madness             mad                 madness             
hilarious           hilari              hilarious           
piece               piec                piece               
multiple            multipl             multiple            
premise             premis              premise             
funny               funni               funny               
barely              bare                barely              

At least five examples are displayed where stemming and lemmatization behave differently.


,Original,Stemmed,Lemmatized
0,inspired,inspir,inspire
1,soontobe,soontob,soontobe
2,feature,featur,feature
3,madness,mad,madness
4,hilarious,hilari,hilarious
5,piece,piec,piece
6,multiple,multipl,multiple
7,premise,premis,premise
8,funny,funni,funny
9,barely,bare,barely


**Observations**

- On the assignment's own example words: `studies` -> lemma `study` (matches the docx table exactly), `running` -> lemma `run`, `computers` -> lemma `computer`. `better` is the most instructive case -- WordNet's adjective lemmatizer maps it to `good` through a small built-in exception list, exactly matching the docx's own note that *"result may depend on POS information and the lemmatizer used."* This only works correctly because `lemmatize_tokens()` looks up each word's POS tag before lemmatizing; lemmatizing without POS information (the default `pos="n"` in a naive call) would leave `better` and `running` unchanged.
- `children` and `mice` lemmatize to their correct irregular singular forms (`child`, `mouse`) because WordNet's dictionary-based lookup handles irregular morphology, which a purely rule-based stemmer cannot do.
- The dataset table above lists real `(original, stemmed, lemmatized)` triples found in the selected sample range (`START_SAMPLE`-`END_SAMPLE`) where the two techniques disagree, satisfying the "at least 5 examples from your own dataset" requirement. In general, expect the stemmed column to contain more truncated, non-dictionary forms than the lemmatized column.

# Task 7 -- Complete Reusable Pipeline

**Assignment requirement:** build a reusable `preprocess_text(text, ...)` function that combines the pipeline stages (segmentation -> tokenization -> cleaning -> normalization -> stopword handling -> stemming/lemmatization), and allow major operations to be independently enabled or disabled.

`preprocess_text()` is the single configurable function the rest of the notebook (and the full-dataset statistics in Task 9) is built on. Every stage has its own boolean switch, and the individual cleaning operations inside it (`remove_html_tags`, `remove_url`, `remove_email`, `remove_punct`, `remove_num`, `normalize_space`) can also be toggled independently. `demonstrate_pipeline_configuration()` proves this by running three different configurations over the same input text, and `process_selected_samples()` then applies the complete pipeline across the whole selected sample range.

In [61]:

def preprocess_text(
    text,
    do_sentence_segmentation=True,
    do_tokenization=True,
    do_cleaning=True,
    do_normalization=True,
    do_stopwords=True,
    do_stemming=True,
    do_lemmatization=False,
    preserve_negation=True,
    remove_html_tags=True,
    remove_url=True,
    remove_email=True,
    remove_punct=True,
    remove_num=True,
    normalize_space=True
):
    """
    Reusable preprocessing function.

    Major stages can be independently enabled/disabled:
    sentence segmentation, tokenization, cleaning, normalization,
    stopword handling, stemming and lemmatization.

    Individual cleaning operations can also be enabled/disabled.
    """
    if do_sentence_segmentation:
        sentences = segment_sentences(text)
    else:
        sentences = [text]

    all_tokens = []

    for sentence in sentences:
        current_text = sentence

        if do_cleaning:
            current_text = clean_text(
                current_text,
                lowercase=do_normalization,
                remove_html_tags=remove_html_tags,
                remove_url=remove_url,
                remove_email=remove_email,
                remove_punct=remove_punct,
                remove_num=remove_num,
                normalize_space=normalize_space
            )
        elif do_normalization:
            current_text = current_text.lower()
            if normalize_space:
                current_text = normalize_whitespace(current_text)

        if do_tokenization:
            tokens = nltk_tokenizer(current_text)
            tokens = filter_word_tokens(tokens)
        else:
            tokens = [current_text] if current_text.strip() else []

        if do_stopwords:
            tokens = remove_stopwords(
                tokens,
                preserve_negation=preserve_negation
            )

        all_tokens.extend(tokens)

    stemmed = stem_tokens(all_tokens) if do_stemming else None
    lemmatized = (
        lemmatize_tokens(all_tokens)
        if do_lemmatization else None
    )

    return {
        "tokens": all_tokens,
        "stemmed": stemmed,
        "lemmatized": lemmatized
    }


def demonstrate_pipeline_configuration():
    """Verify that major preprocessing operations can be toggled."""
    print_header("PIPELINE CONFIGURATION TEST")

    sample = (
        "This is NOT a useful example! "
        "Visit https://example.com and call 123."
    )

    print(
        "preprocess_text() supports independent enable/disable controls "
        "for the major preprocessing stages."
    )

    print("\nConfiguration A - full preprocessing:")
    print(preprocess_text(
        sample,
        do_sentence_segmentation=True,
        do_tokenization=True,
        do_cleaning=True,
        do_normalization=True,
        do_stopwords=True,
        do_stemming=True,
        do_lemmatization=True,
        preserve_negation=True
    ))

    print("\nConfiguration B - cleaning, stopwords and stemming disabled:")
    print(preprocess_text(
        sample,
        do_sentence_segmentation=True,
        do_tokenization=True,
        do_cleaning=False,
        do_normalization=False,
        do_stopwords=False,
        do_stemming=False,
        do_lemmatization=False
    ))

    print("\nConfiguration C - keep numbers while cleaning other noise:")
    print(preprocess_text(
        sample,
        do_sentence_segmentation=True,
        do_tokenization=True,
        do_cleaning=True,
        do_normalization=True,
        do_stopwords=True,
        do_stemming=False,
        do_lemmatization=False,
        preserve_negation=True,
        remove_num=False
    ))

    print(
        "\nObservation: major stages and individual cleaning operations "
        "can be enabled or disabled without rewriting the pipeline."
    )
    print_success("Pipeline configuration controls verified.")


demonstrate_pipeline_configuration()


PIPELINE CONFIGURATION TEST
preprocess_text() supports independent enable/disable controls for the major preprocessing stages.

Configuration A - full preprocessing:
{'tokens': ['not', 'useful', 'example', 'visit', 'call'], 'stemmed': ['not', 'use', 'exampl', 'visit', 'call'], 'lemmatized': ['not', 'useful', 'example', 'visit', 'call']}

Configuration B - cleaning, stopwords and stemming disabled:
{'tokens': ['This', 'is', 'NOT', 'a', 'useful', 'example', 'Visit', 'https', '//example.com', 'and', 'call'], 'stemmed': None, 'lemmatized': None}

Configuration C - keep numbers while cleaning other noise:
{'tokens': ['not', 'useful', 'example', 'visit', 'call'], 'stemmed': None, 'lemmatized': None}

Observation: major stages and individual cleaning operations can be enabled or disabled without rewriting the pipeline.

[SUCCESS] Pipeline configuration controls verified.


In [62]:


def demonstrate_complete_pipeline(
    selected_df
):

    print_header(
        "STEP 10 - COMPLETE PREPROCESSING PIPELINE"
    )

    print(
        "Pipeline:"
    )

    print(
        "Raw Text"
    )

    print(
        "   \u2193"
    )

    print(
        "Sentence Segmentation"
    )

    print(
        "   \u2193"
    )

    print(
        "Tokenization"
    )

    print(
        "   \u2193"
    )

    print(
        "Cleaning"
    )

    print(
        "   \u2193"
    )

    print(
        "Normalization"
    )

    print(
        "   \u2193"
    )

    print(
        "Stopword Handling"
    )

    print(
        "   \u2193"
    )

    print(
        "Stemming + Lemmatization"
    )

    print(
        "   \u2193"
    )

    print(
        "Final Preprocessed Representation"
    )

    if len(selected_df) == 0:

        return

    sample_text = (
        selected_df.iloc[0]["review"]
    )

    result = preprocess_text(
        sample_text,
        do_sentence_segmentation=True,
        do_tokenization=True,
        do_cleaning=True,
        do_normalization=True,
        do_stopwords=True,
        do_stemming=True,
        do_lemmatization=True,
        preserve_negation=True
    )

    print_subheader(
        "PIPELINE OUTPUT FOR ONE DOCUMENT"
    )

    print(
        "\nOriginal text:"
    )

    print(
        sample_text[:1000]
    )

    print(
        "\nFinal tokens:"
    )

    print(
        display_tokens(
            result["tokens"]
        )
    )

    print(
        "\nStemmed tokens:"
    )

    print(
        display_tokens(
            result["stemmed"]
        )
    )

    print(
        "\nLemmatized tokens:"
    )

    print(
        display_tokens(
            result["lemmatized"]
        )
    )

    print_success(
        "STEP 10 completed successfully."
    )


demonstrate_complete_pipeline(selected_df)


STEP 10 - COMPLETE PREPROCESSING PIPELINE
Pipeline:
Raw Text
   ↓
Sentence Segmentation
   ↓
Tokenization
   ↓
Cleaning
   ↓
Normalization
   ↓
Stopword Handling
   ↓
Stemming + Lemmatization
   ↓
Final Preprocessed Representation

----------------------------------------------------------------------
PIPELINE OUTPUT FOR ONE DOCUMENT
----------------------------------------------------------------------

Original text:
This short film that inspired the soon-to-be full length feature - Spatula Madness - is a hilarious piece that contends against similar cartoons yielding multiple writers. The short film stars Edward the Spatula who after being fired from his job, joins in the fight against the evil spoons. This premise allows for some funny content near the beginning, but is barely present for the remainder of the feature. This film's 15-minute running time is absorbed by some odd-ball comedy and a small musical number. Unfortunately not much else lies below it. The plot that is set up

In [63]:

def process_selected_samples(
    selected_df,
    start
):

    print_header(
        "STEP 11 - PROCESS SELECTED DATASET SAMPLES"
    )

    processed_rows = []

    for position in range(
        len(selected_df)
    ):

        row = selected_df.iloc[position]

        result = preprocess_text(
            row["review"],
            do_sentence_segmentation=True,
            do_tokenization=True,
            do_cleaning=True,
            do_normalization=True,
            do_stopwords=True,
            do_stemming=True,
            do_lemmatization=True,
            preserve_negation=True
        )

        processed_rows.append(
            {
                "sample_number":
                    start + position,

                "sentiment":
                    row["sentiment"],

                "original":
                    row["review"],

                "tokens":
                    result["tokens"],

                "stemmed":
                    result["stemmed"],

                "lemmatized":
                    result["lemmatized"]
            }
        )

    print(
        f"Documents processed: "
        f"{len(processed_rows)}"
    )

    print_success(
        "STEP 11 completed successfully."
    )

    return processed_rows


processed_samples = process_selected_samples(selected_df, start)


STEP 11 - PROCESS SELECTED DATASET SAMPLES
Documents processed: 100

[SUCCESS] STEP 11 completed successfully.


**Observations**

- Configuration A (everything on) shows the fully cleaned, stemmed, and lemmatized output; Configuration B (segmentation/tokenization only, no cleaning/stopwords/stemming) shows the pipeline can also be used just to tokenize raw text; Configuration C independently toggles one cleaning switch (`remove_num=False`) while leaving the other stages on -- together the three runs demonstrate the "easily enable or disable major preprocessing operations" requirement directly.
- Configuration C is also a useful illustration of how the pipeline's stages interact: `remove_num=False` changes what `clean_text()` does (digits are no longer stripped from the text), but tokenization applies `filter_word_tokens()` afterward, which keeps only tokens containing at least one letter -- so a token that is *purely* numeric (like a standalone "123") is still dropped at the tokenization stage even though `remove_num=False` kept it through cleaning. A number embedded in an alphanumeric token (e.g. "10/10" surviving as "1010" style text) would behave differently. This is a good example of why it matters which stage a setting belongs to, not just whether a stage is "on".
- Note `do_lemmatization` defaults to `False` in `preprocess_text()`'s signature (stemming is the default reducer), but every demonstration in this notebook that reports lemmatized output explicitly passes `do_lemmatization=True` -- an example of a sensible default that remains fully overridable per call.
- `process_selected_samples()` confirms the complete pipeline runs cleanly across the whole selected range (`START_SAMPLE`-`END_SAMPLE`), not just on a single document -- these processed rows are what Task 9's full-dataset statistics build on at scale.

# Task 8 -- Before vs After Transformation

**Assignment requirement:** select 5 documents and show their transformation through the pipeline: **Original -> Tokenized -> Normalized -> Stemmed -> Lemmatized**.

`demonstrate_five_transformations()` (STEP 13) walks the first `TRANSFORMATION_DOCUMENTS` (5) documents of the **full dataset** through each stage in turn, printing every intermediate representation. A compact DataFrame summary follows for a side-by-side view of all 5 documents at once.

In [64]:


def demonstrate_five_transformations(
    df
):

    print_header(
        "STEP 13 - SAMPLE TRANSFORMATION DEMONSTRATION"
    )

    print(
        "Five documents are transformed through:"
    )

    print(
        "Original \u2192 Tokenized \u2192 Normalized "
        "\u2192 Stemmed \u2192 Lemmatized"
    )

    number_of_documents = min(
        TRANSFORMATION_DOCUMENTS,
        len(df)
    )

    for position in range(
        number_of_documents
    ):

        text = df.iloc[
            position
        ]["review"]

        print_subheader(
            f"DOCUMENT {position + 1}"
        )


        print(
            "\nORIGINAL TEXT:"
        )

        print(
            text[:800]
        )

        if len(text) > 800:
            print("...")


        tokens = nltk_tokenizer(
            text
        )

        print(
            "\nTOKENIZED TEXT:"
        )

        print(
            display_tokens(
                tokens
            )
        )



        normalized = clean_text(
            text,
            lowercase=True,
            remove_html_tags=True,
            remove_url=True,
            remove_email=True,
            remove_punct=True,
            remove_num=True,
            normalize_space=True
        )

        print(
            "\nNORMALIZED TEXT:"
        )

        print(
            normalized[:800]
        )

        if len(normalized) > 800:
            print("...")


        normalized_tokens = filter_word_tokens(
            nltk_tokenizer(
                normalized
            )
        )

        normalized_tokens = remove_stopwords(
            normalized_tokens,
            preserve_negation=True
        )

        stems = stem_tokens(
            normalized_tokens
        )

        print(
            "\nSTEMMED TEXT:"
        )

        print(
            display_tokens(
                stems
            )
        )



        lemmas = lemmatize_tokens(
            normalized_tokens
        )

        print(
            "\nLEMMATIZED TEXT:"
        )

        print(
            display_tokens(
                lemmas
            )
        )

    print_success(
        "STEP 13 completed successfully."
    )


demonstrate_five_transformations(df)


STEP 13 - SAMPLE TRANSFORMATION DEMONSTRATION
Five documents are transformed through:
Original → Tokenized → Normalized → Stemmed → Lemmatized

----------------------------------------------------------------------
DOCUMENT 1
----------------------------------------------------------------------

ORIGINAL TEXT:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass f

In [65]:


def _truncate(value, width=90):
    text = str(value)
    return text if len(text) <= width else text[:width] + "..."


_five_doc_rows = []

for _position in range(min(TRANSFORMATION_DOCUMENTS, len(df))):

    _text = df.iloc[_position]["review"]
    _tokens = nltk_tokenizer(_text)

    _normalized = clean_text(
        _text,
        lowercase=True,
        remove_html_tags=True,
        remove_url=True,
        remove_email=True,
        remove_punct=True,
        remove_num=True,
        normalize_space=True
    )

    _normalized_tokens = filter_word_tokens(nltk_tokenizer(_normalized))
    _normalized_tokens = remove_stopwords(_normalized_tokens, preserve_negation=True)

    _stems = stem_tokens(_normalized_tokens)
    _lemmas = lemmatize_tokens(_normalized_tokens)

    _five_doc_rows.append({
        "Document": _position + 1,
        "Original": _truncate(_text),
        "Tokenized": _truncate(_tokens),
        "Normalized": _truncate(_normalized),
        "Stemmed": _truncate(_stems),
        "Lemmatized": _truncate(_lemmas),
    })

five_transformations_df = pd.DataFrame(_five_doc_rows)
display(five_transformations_df)

,Document,Original,Tokenized,Normalized,Stemmed,Lemmatized
0,1,One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be h...,"['One', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching'...",one of the other reviewers has mentioned that after watching just oz episode youll be hook...,"['one', 'review', 'mention', 'watch', 'oz', 'episod', 'youll', 'hook', 'right', 'exactli',...","['one', 'reviewer', 'mention', 'watch', 'oz', 'episode', 'youll', 'hook', 'right', 'exactl..."
1,2,A wonderful little production. <br /><br />The filming technique is very unassuming- very ...,"['A', 'wonderful', 'little', 'production', '.', '<', 'br', '/', '>', '<', 'br', '/', '>', ...",a wonderful little production the filming technique is very unassuming very oldtimebbc fas...,"['wonder', 'littl', 'product', 'film', 'techniqu', 'unassum', 'oldtimebbc', 'fashion', 'gi...","['wonderful', 'little', 'production', 'film', 'technique', 'unassuming', 'oldtimebbc', 'fa..."
2,3,"I thought this was a wonderful way to spend time on a too hot summer weekend, sitting in t...","['I', 'thought', 'this', 'was', 'a', 'wonderful', 'way', 'to', 'spend', 'time', 'on', 'a',...",i thought this was a wonderful way to spend time on a too hot summer weekend sitting in th...,"['thought', 'wonder', 'way', 'spend', 'time', 'hot', 'summer', 'weekend', 'sit', 'air', 'c...","['think', 'wonderful', 'way', 'spend', 'time', 'hot', 'summer', 'weekend', 'sit', 'air', '..."
3,4,Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet...,"['Basically', 'there', ""'s"", 'a', 'family', 'where', 'a', 'little', 'boy', '(', 'Jake', ')...",basically theres a family where a little boy jake thinks theres a zombie in his closet his...,"['basic', 'there', 'famili', 'littl', 'boy', 'jake', 'think', 'there', 'zombi', 'closet', ...","['basically', 'there', 'family', 'little', 'boy', 'jake', 'think', 'theres', 'zombie', 'cl..."
4,5,"Petter Mattei's ""Love in the Time of Money"" is a visually stunning film to watch. Mr. Matt...","['Petter', 'Mattei', ""'s"", '``', 'Love', 'in', 'the', 'Time', 'of', 'Money', ""''"", 'is', '...",petter matteis love in the time of money is a visually stunning film to watch mr mattei of...,"['petter', 'mattei', 'love', 'time', 'money', 'visual', 'stun', 'film', 'watch', 'mr', 'ma...","['petter', 'matteis', 'love', 'time', 'money', 'visually', 'stunning', 'film', 'watch', 'm..."


**Observations**

- Each of the 5 documents follows the identical five-stage path (Original -> Tokenized -> Normalized -> Stemmed -> Lemmatized), which is exactly what the assignment's Task 8 asks to be shown -- the DataFrame above gives a compact side-by-side comparison, and the printed output above it gives the full, untruncated detail for each stage.
- Token counts shrink noticeably from "Tokenized" to "Stemmed"/"Lemmatized" once stopwords are removed, but the *number of distinct word forms* does not necessarily shrink at the same rate -- stemming can merge several original words onto the same stem (e.g. every member of a word family collapsing onto one root, as seen in Task 5).
- "Stemmed" and "Lemmatized" are computed from the same `normalized_tokens` in each document, so any word where the two columns disagree here is the same kind of case explored more systematically in Task 6.

# Task 9 -- Dataset Statistics

**Assignment requirement (part of the assignment's Task 8, "Before vs. After Analysis"):** report, for the complete dataset, total tokens, unique tokens, average tokens/document, and vocabulary size before vs. after preprocessing, and briefly explain why the vocabulary size changed.

`calculate_before_statistics()` measures the raw dataset using the same `basic_tokens()` regex tokenizer used in Task 1 and Dataset Loading. `calculate_after_statistics()` runs the **complete** `preprocess_text()` pipeline (cleaning, tokenization, stopword removal, Porter stemming) over all 50,000 reviews.

> **Runtime note:** `calculate_after_statistics()` runs the full NLTK-based pipeline over all 50,000 reviews, so this cell can take a few minutes on Colab's default CPU runtime. It prints progress every 5,000 documents so you can see it is working; this is unchanged from the original script and is not simplified or sampled down, per the requirement to preserve the original implementation exactly.

In [66]:


def calculate_before_statistics(
    df
):

    total_tokens = 0

    vocabulary = set()

    for text in df["review"]:

        tokens = basic_tokens(
            text
        )

        total_tokens += len(tokens)

        vocabulary.update(
            tokens
        )

    average = (
        total_tokens / len(df)
        if len(df) > 0
        else 0
    )

    return {
        "total_tokens":
            total_tokens,

        "unique_tokens":
            len(vocabulary),

        "average_tokens":
            average,

        "vocabulary_size":
            len(vocabulary)
    }


def calculate_after_statistics(
    df
):

    total_tokens = 0

    vocabulary = set()

    for index, text in enumerate(
        df["review"],
        start=1
    ):

        result = preprocess_text(
            text,
            do_sentence_segmentation=False,
            do_tokenization=True,
            do_cleaning=True,
            do_normalization=True,
            do_stopwords=True,
            do_stemming=True,
            do_lemmatization=False,
            preserve_negation=True
        )

        tokens = result["stemmed"]

        if tokens is None:
            tokens = []

        total_tokens += len(tokens)

        vocabulary.update(
            tokens
        )

        if index % 5000 == 0:

            print(
                f"Processed "
                f"{index}/{len(df)} documents..."
            )

    average = (
        total_tokens / len(df)
        if len(df) > 0
        else 0
    )

    return {
        "total_tokens":
            total_tokens,

        "unique_tokens":
            len(vocabulary),

        "average_tokens":
            average,

        "vocabulary_size":
            len(vocabulary)
    }


def display_statistics_table(
    before,
    after
):

    print_header(
        "STEP 12 - BEFORE VS AFTER ANALYSIS"
    )

    print(
        f"{'Measure':<30}"
        f"{'Before Preprocessing':>22}"
        f"{'After Preprocessing':>22}"
    )

    print(
        "-" * 74
    )

    print(
        f"{'Total tokens':<30}"
        f"{before['total_tokens']:>22}"
        f"{after['total_tokens']:>22}"
    )

    print(
        f"{'Unique tokens':<30}"
        f"{before['unique_tokens']:>22}"
        f"{after['unique_tokens']:>22}"
    )

    print(
        f"{'Average tokens/document':<30}"
        f"{before['average_tokens']:>22.2f}"
        f"{after['average_tokens']:>22.2f}"
    )

    print(
        f"{'Vocabulary size':<30}"
        f"{before['vocabulary_size']:>22}"
        f"{after['vocabulary_size']:>22}"
    )

    reduction = (
        (
            before["vocabulary_size"]
            - after["vocabulary_size"]
        )
        / before["vocabulary_size"]
        * 100
    )

    print(
        f"\nVocabulary reduction: "
        f"{reduction:.2f}%"
    )

    print(
        "\nExplanation:"
    )

    print("Correct interpretation:")
    print(
        "The total token count and average tokens per document decrease "
        "because cleaning, stopword removal and normalization remove or "
        "consolidate text."
    )
    print(
        "Vocabulary size does not have to decrease. In this implementation "
        "the BEFORE vocabulary is built from basic raw-text tokens, while "
        "the AFTER vocabulary is built from cleaned, stopword-filtered and "
        "Porter-stemmed tokens. These are different token representations."
    )
    print(
        "Therefore, the observed vocabulary may decrease or increase. "
        "An increase does not mean preprocessing failed; it means the "
        "before/after vocabularies are not directly identical token spaces."
    )
    print(
        "Cleaning can split HTML fragments and normalize attached text, "
        "while stemming can create stem forms that were not present as "
        "exact raw tokens. Vocabulary should therefore be reported as a "
        "change rather than automatically called a reduction."
    )

    print_success(
        "STEP 12 completed successfully."
    )


print("Calculating BEFORE preprocessing statistics for complete dataset...")
before_stats = calculate_before_statistics(df)

print("\nCalculating AFTER preprocessing statistics for complete dataset...")
after_stats = calculate_after_statistics(df)

display_statistics_table(before_stats, after_stats)

Calculating BEFORE preprocessing statistics for complete dataset...

Calculating AFTER preprocessing statistics for complete dataset...
Processed 5000/50000 documents...
Processed 10000/50000 documents...
Processed 15000/50000 documents...
Processed 20000/50000 documents...
Processed 25000/50000 documents...
Processed 30000/50000 documents...
Processed 35000/50000 documents...
Processed 40000/50000 documents...
Processed 45000/50000 documents...
Processed 50000/50000 documents...

STEP 12 - BEFORE VS AFTER ANALYSIS
Measure                         Before Preprocessing   After Preprocessing
--------------------------------------------------------------------------
Total tokens                                11748522               6044854
Unique tokens                                 112358                127195
Average tokens/document                       234.97                120.90
Vocabulary size                               112358                127195

Vocabulary reduction: -13.21

In [67]:


def _format_measure(value, is_average):

    if is_average:
        return f"{value:.2f}"
    return f"{value:,}"


before_after_df = pd.DataFrame({
    "Measure": [
        "Total tokens",
        "Unique tokens",
        "Average tokens/document",
        "Vocabulary size",
    ],
    "Before Preprocessing": [
        _format_measure(before_stats["total_tokens"], False),
        _format_measure(before_stats["unique_tokens"], False),
        _format_measure(before_stats["average_tokens"], True),
        _format_measure(before_stats["vocabulary_size"], False),
    ],
    "After Preprocessing": [
        _format_measure(after_stats["total_tokens"], False),
        _format_measure(after_stats["unique_tokens"], False),
        _format_measure(after_stats["average_tokens"], True),
        _format_measure(after_stats["vocabulary_size"], False),
    ],
})

display(before_after_df)

,Measure,Before Preprocessing,After Preprocessing
0,Total tokens,"11,748,522","6,044,854"
1,Unique tokens,"112,358","127,195"
2,Average tokens/document,234.97,120.90
3,Vocabulary size,"112,358","127,195"


**Observations**

- Total tokens and average tokens/document both drop after preprocessing, because stopword removal deletes a large fraction of tokens (function words like "the", "a", "and" are extremely frequent) and cleaning removes HTML/punctuation/number tokens entirely.
- **Why vocabulary size changes (the assignment's specific question):** the *before* vocabulary is built from raw, unstemmed, case-sensitive-free but otherwise unprocessed word tokens (`basic_tokens()`), while the *after* vocabulary is built from cleaned, stopword-filtered, **Porter-stemmed** tokens (`preprocess_text(..., do_stemming=True)`). Stemming is many-to-one (e.g. "loved", "loving", "loves" all collapse to `love`), which pushes vocabulary size down; at the same time, HTML/punctuation stripping can split previously-joined character sequences into new standalone tokens, which pushes it back up. The net direction depends on which effect dominates for this dataset -- the table above reports the actual observed result rather than assuming it must be a reduction.
- This before/after comparison is only meaningful because both statistics were computed the same way for every one of the 50,000 documents -- which is why this cell processes the full dataset rather than only the selected sample range.

# Task 10 -- Aggressive vs Meaning-Preserving Clinical Pipeline

**Assignment requirement:** for the sentence *"The patient is not suffering from diabetes, but his glucose level increased to 180 mg/dL,"* build an aggressive pipeline (remove stopwords, punctuation, numbers, apply stemming) and a meaning-preserving pipeline, compare the outputs, state what information Pipeline A loses, and explain why blindly applying traditional preprocessing can reduce NLP system quality.

`aggressive_clinical_pipeline()` reuses the same building blocks as the rest of the notebook (`remove_punctuation`, `remove_numbers`, `nltk_tokenizer`, `remove_stopwords` with `preserve_negation=False`, `stem_tokens`) with nothing held back. `meaning_preserving_clinical_pipeline()` instead keeps negation words, numeric values, and clinical units (like `mg/dL`) intact.

In [68]:


def aggressive_clinical_pipeline(
    text
):


    text = text.lower()

    text = remove_punctuation(
        text
    )

    text = remove_numbers(
        text
    )

    tokens = nltk_tokenizer(
        text
    )

    tokens = remove_stopwords(
        tokens,
        preserve_negation=False
    )

    tokens = stem_tokens(
        tokens
    )

    return tokens


def meaning_preserving_clinical_pipeline(
    text
):

    text = text.lower()

    text = normalize_whitespace(
        text
    )


    tokens = nltk_tokenizer(
        text
    )

    meaningful_tokens = []

    for token in tokens:

        if re.match(
            r"^\d+(?:\.\d+)?$",
            token
        ):

            meaningful_tokens.append(
                token
            )

        elif re.match(
            r"^\d+(?:\.\d+)?/[A-Za-z]+$",
            token
        ):

            meaningful_tokens.append(
                token
            )

        elif re.search(
            r"[A-Za-z]",
            token
        ):

            cleaned = re.sub(
                r"^[^\w]+|[^\w]+$",
                "",
                token
            )

            if cleaned:

                meaningful_tokens.append(
                    cleaned
                )

    stop_words = get_stopwords()

    stop_words -= {
        "not",
        "no",
        "nor",
        "never"
    }

    final_tokens = [
        token
        for token in meaningful_tokens
        if token.lower()
        not in stop_words
        or token.lower()
        in {"not", "no", "nor", "never"}
    ]

    return final_tokens


def demonstrate_clinical_pipelines():

    print_header(
        "STEP 14 - AGGRESSIVE VS MEANING-PRESERVING PIPELINE"
    )

    clinical_sentence = (
        "The patient is not suffering from diabetes, "
        "but his glucose level increased to 180 mg/dL."
    )

    print(
        "\nClinical sentence:"
    )

    print(
        clinical_sentence
    )

    aggressive = aggressive_clinical_pipeline(
        clinical_sentence
    )

    meaning_preserving = (
        meaning_preserving_clinical_pipeline(
            clinical_sentence
        )
    )

    print_subheader(
        "PIPELINE A - AGGRESSIVE PREPROCESSING"
    )

    print(
        "Operations:"
    )

    print(
        "- Remove stopwords"
    )

    print(
        "- Remove punctuation"
    )

    print(
        "- Remove numbers"
    )

    print(
        "- Apply stemming"
    )

    print(
        "\nOutput:"
    )

    print(
        aggressive
    )

    print_subheader(
        "PIPELINE B - MEANING-PRESERVING PREPROCESSING"
    )

    print(
        "Important clinical information is preserved:"
    )

    print(
        "- Negation"
    )

    print(
        "- Numeric glucose value"
    )

    print(
        "- Clinical unit"
    )

    print(
        "- Important medical terminology"
    )

    print(
        "\nOutput:"
    )

    print(
        meaning_preserving
    )

    print_subheader(
        "CRITICAL ANALYSIS"
    )

    print(
        "Important information lost by Pipeline A:"
    )

    print(
        "1. The word 'not' can be removed, "
        "changing the meaning from absence of "
        "diabetes to possible diabetes."
    )

    print(
        "2. The number 180 is removed, losing "
        "the actual glucose measurement."
    )

    print(
        "3. The unit mg/dL may be damaged or "
        "removed, making the numerical value "
        "less clinically meaningful."
    )

    print(
        "4. Stemming can produce unnatural "
        "word forms."
    )

    print(
        "\nWhy blind preprocessing can reduce "
        "NLP quality:"
    )

    print(
        "Traditional preprocessing assumes that "
        "all punctuation, stopwords and numbers "
        "are noise. In clinical text, however, "
        "negation, numerical measurements, units "
        "and terminology can carry the central "
        "meaning of the sentence."
    )

    print_success(
        "STEP 14 completed successfully."
    )


demonstrate_clinical_pipelines()


STEP 14 - AGGRESSIVE VS MEANING-PRESERVING PIPELINE

Clinical sentence:
The patient is not suffering from diabetes, but his glucose level increased to 180 mg/dL.

----------------------------------------------------------------------
PIPELINE A - AGGRESSIVE PREPROCESSING
----------------------------------------------------------------------
Operations:
- Remove stopwords
- Remove punctuation
- Remove numbers
- Apply stemming

Output:
['patient', 'suffer', 'diabet', 'glucos', 'level', 'increas', 'mgdl']

----------------------------------------------------------------------
PIPELINE B - MEANING-PRESERVING PREPROCESSING
----------------------------------------------------------------------
Important clinical information is preserved:
- Negation
- Numeric glucose value
- Clinical unit
- Important medical terminology

Output:
['patient', 'not', 'suffering', 'diabetes', 'glucose', 'level', 'increased', '180', 'mg/dl']

----------------------------------------------------------------------


**Observations**

- **What Pipeline A loses:** the negation word "not" (`preserve_negation=False`), the exact glucose measurement `180`, and the clinical unit `mg/dL` (its slash and letters are treated as ordinary punctuation/tokens and can be broken apart by aggressive punctuation removal). Losing "not" is the most dangerous: it can flip the sentence from *"does not have diabetes"* to a bag of words that reads as if diabetes were being reported directly.
- **Why blind preprocessing can reduce NLP system quality:** standard preprocessing heuristics (strip punctuation, strip numbers, strip all stopwords, stem everything) were designed with general-purpose text and classic information-retrieval use cases in mind, where exact numbers and negation matter less than topical keyword overlap. Clinical, legal, and financial text is different -- negation changes diagnosis, numbers are measurements, and units disambiguate them. A pipeline that is appropriate for one domain (e.g. the IMDB sentiment pipeline used throughout this notebook, which also preserves negation for the same underlying reason) is not automatically safe for another; preprocessing decisions should always be justified against what the downstream task actually needs, which is precisely the point this task is designed to illustrate.
- Note that the **main IMDB pipeline in this notebook already applies the same principle**: `preserve_negation=True` is the default throughout Tasks 4-9, for exactly the reason demonstrated more explicitly here for clinical text.

## Assignment Compliance Check

| Assignment Deliverable | Notebook Section |
|---|---|
| Dataset selection (real-world text, >= 50 samples, sufficient variation) | Introduction -- Dataset Description; Dataset Loading |
| Task 1 -- number of documents, sample documents, average length, vocabulary, noise types | Dataset Loading (quick preview) and Task 1 -- Dataset Exploration |
| Task 2 -- sentence segmentation on >= 3 documents; abbreviation discussion | Task 2 -- Sentence Segmentation |
| Task 3 -- >= 2 tokenizers compared; >= 3 differing cases identified | Task 3 -- Tokenization Comparison (3 tokenizers compared) |
| Task 4 -- lowercase / whitespace / punctuation / URLs / emails / special characters / numbers / stopwords, with justified decisions | Task 4 -- Cleaning and Normalization |
| Task 5 -- stemming algorithm, worked-example table, dataset examples of unnatural stems | Task 5 -- Stemming |
| Task 6 -- lemmatization, original/stemmed/lemmatized comparison, >= 5 dataset examples | Task 6 -- Lemmatization |
| Task 7 -- reusable `preprocess_text()` with enable/disable controls | Task 7 -- Complete Reusable Pipeline |
| Task 8 -- 5-document Original -> Tokenized -> Normalized -> Stemmed -> Lemmatized walkthrough | Task 8 -- Before vs After Transformation |
| Task 8 (continued) -- before/after totals, unique tokens, average tokens/doc, vocabulary size for the complete dataset; explanation of the vocabulary-size change | Task 9 -- Dataset Statistics |
| Task 10 -- aggressive vs. meaning-preserving clinical pipelines; what is lost; why blind preprocessing can reduce NLP quality | Task 10 -- Aggressive vs Meaning-Preserving Clinical Pipeline |
| Deliverable: Jupyter Notebook with dataset loading, implementation, intermediate outputs, final pipeline, tables/observations | This notebook, top to bottom |
| Deliverable: dataset description | Introduction -- Dataset Description |
| Deliverable: preprocessing decisions (justified) | Task 4 Observations; Task 10 Observations |
| Deliverable: stemming vs. lemmatization observations | Task 5 and Task 6 Observations |
| Deliverable: before/after analysis | Task 9 Observations |
| Deliverable: answers to critical-analysis questions | Task 9 Observations (vocabulary change); Task 10 Observations (information loss, blind preprocessing) |
| Deliverable: dataset/source reference | Dataset Reference (below) |

## Final Observations

`final_summary()`  prints a completion checklist confirming every pipeline stage ran successfully.

In [69]:


def final_summary():

    print_header(
        "STEP 15 - FINAL ANALYSIS"
    )

    print(
        "The NLP preprocessing pipeline has "
        "implemented the required stages."
    )

    print(
        "\nCompleted stages:"
    )

    stages = [
        "Dataset selection and validation",
        "Dataset loading",
        "Dataset exploration",
        "Sentence segmentation",
        "Tokenization comparison",
        "Text normalization and cleaning",
        "Stopword handling",
        "Stemming",
        "Lemmatization",
        "Complete preprocessing pipeline with configurable operations",
        "Selected dataset processing",
        "Before vs after statistics",
        "Five-document transformation",
        "Aggressive vs meaning-preserving analysis"
    ]

    for number, stage in enumerate(
        stages,
        start=1
    ):

        print(
            f"[SUCCESS] {stage}"
        )

    print(
        "\nThe final representation is suitable "
        "for subsequent NLP tasks such as "
        "classification, sentiment analysis, "
        "feature extraction and machine learning."
    )

    print_success(
        "NLP preprocessing practical completed."
    )


final_summary()


STEP 15 - FINAL ANALYSIS
The NLP preprocessing pipeline has implemented the required stages.

Completed stages:
[SUCCESS] Dataset selection and validation
[SUCCESS] Dataset loading
[SUCCESS] Dataset exploration
[SUCCESS] Sentence segmentation
[SUCCESS] Tokenization comparison
[SUCCESS] Text normalization and cleaning
[SUCCESS] Stopword handling
[SUCCESS] Stemming
[SUCCESS] Lemmatization
[SUCCESS] Complete preprocessing pipeline with configurable operations
[SUCCESS] Selected dataset processing
[SUCCESS] Before vs after statistics
[SUCCESS] Five-document transformation
[SUCCESS] Aggressive vs meaning-preserving analysis

The final representation is suitable for subsequent NLP tasks such as classification, sentiment analysis, feature extraction and machine learning.

[SUCCESS] NLP preprocessing practical completed.


### Summary of Key Findings

- **Noise and cleaning:** the IMDB reviews contain HTML tags (~58% of documents), punctuation, numbers/ratings, contractions, and inconsistent capitalization. `clean_text()` addresses all of these, with every operation independently toggleable so the pipeline is not "blindly" removing information it might later need.
- **Negation is preserved by design:** because this is a sentiment dataset, `not`, `no`, `nor`, and `never` are kept through stopword removal by default (`preserve_negation=True`). Task 10 shows concretely what is lost when a pipeline does *not* make this choice.
- **Stemming vs. lemmatization:** Porter stemming is fast and rule-based but can produce non-dictionary forms (e.g. `studies` -> `studi`); WordNet lemmatization is slower but POS-aware and dictionary-grounded, correctly handling irregular forms (`better` -> `good`, `children` -> `child`). Both are implemented, and both are available as independent, toggleable options inside `preprocess_text()`.
- **Vocabulary size is not guaranteed to shrink:** Task 9 shows that whether the "after" vocabulary is smaller than the "before" vocabulary depends on the balance between stemming's many-to-one merging effect and cleaning's tendency to reveal new standalone tokens -- the notebook reports the actual measured direction rather than assuming one.
- **Domain matters:** the same "remove stopwords / punctuation / numbers, then stem" recipe that is reasonable for movie-review sentiment analysis would destroy clinically important information (negation, dosages, units) if applied unmodified to clinical text, as demonstrated directly in Task 10.

### Conclusion

This notebook implements a complete, configurable NLP text-preprocessing pipeline over the IMDB Movie Reviews dataset, covering every stage the assignment requires: dataset exploration, sentence segmentation, tokenization comparison, cleaning/normalization, stopword handling, stemming, lemmatization, a single reusable `preprocess_text()` function with independently toggleable stages, before/after dataset-wide statistics, and a critical comparison of aggressive vs. meaning-preserving preprocessing on domain-sensitive text.

The resulting token/stem/lemma representations are ready for downstream NLP work such as sentiment classification, feature extraction (bag-of-words, TF-IDF, embeddings), or information retrieval. The most important overall takeaway from the notebook is that **preprocessing is not one-size-fits-all**: every cleaning and normalization decision made here (keeping negation words, stemming vs. lemmatizing, which characters to strip) was justified against what a sentiment-analysis system built on this specific dataset actually needs, and Task 10 shows explicitly why the same choices would be inappropriate for a different domain.

### Dataset Reference

**Dataset:** IMDB Dataset of 50,000 Movie Reviews (Large Movie Review Dataset / "aclImdb")

**Primary citation:**
Maas, A. L., Daly, R. E., Pham, P. T., Huang, D., Ng, A. Y., & Potts, C. (2011). *Learning Word Vectors for Sentiment Analysis.* Proceedings of the 49th Annual Meeting of the Association for Computational Linguistics: Human Language Technologies, pp. 142-150. Portland, Oregon, USA: Association for Computational Linguistics. https://aclanthology.org/P11-1015/

**Common public sources for this CSV release:**
- Stanford AI Lab (original release): https://ai.stanford.edu/~amaas/data/sentiment/
- Kaggle "IMDB Dataset of 50K Movie Reviews" : https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
